# MCP Tool Test — No LLM

Direct test of the `parametric_shape_generator` MCP tool via the Swiftlet server.  
No LLM involved — tool calls are constructed manually.

**What the tool does:**  
Generates a parametric building footprint (closed polyline) in Rhino/Grasshopper given a shape type and dimensional parameters.

**Steps:**
1. Setup — imports & paths
2. Load config from `mcp.json`
3. Connect to MCP server & discover tools
4. Call `parametric_shape_generator` with default parameters
5. Call `parametric_shape_generator` with custom parameters
6. Batch test — all shape types

## 1. Setup — Imports & Path Configuration

In [1]:
import sys, os, json
from pathlib import Path

# ── Ensure team_04/PY is importable ──────────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

REPO_ROOT    = NOTEBOOK_DIR.parents[1]
ENV_PATH     = REPO_ROOT / ".env"
MCP_CFG_PATH = REPO_ROOT / "mcp.json"

print(f"Repo root : {REPO_ROOT}")
print(f".env      : {ENV_PATH}  (exists={ENV_PATH.exists()})")
print(f"mcp.json  : {MCP_CFG_PATH}  (exists={MCP_CFG_PATH.exists()})")

Repo root : C:\Users\tuemi\Downloads\Glabtools\IAAC Repo\bimsc26-datamgmt-session03\AIA26_Studio
.env      : C:\Users\tuemi\Downloads\Glabtools\IAAC Repo\bimsc26-datamgmt-session03\AIA26_Studio\.env  (exists=True)
mcp.json  : C:\Users\tuemi\Downloads\Glabtools\IAAC Repo\bimsc26-datamgmt-session03\AIA26_Studio\mcp.json  (exists=True)


## 2. Load MCP Endpoint from `mcp.json`

In [2]:
with open(MCP_CFG_PATH, encoding="utf-8") as f:
    mcp_cfg = json.load(f)

servers    = mcp_cfg["mcpServers"]
server_key = next(iter(servers))
server_cfg = servers[server_key]

# Endpoint is in "url" or first element of "args"
MCP_ENDPOINT = server_cfg.get("url") or server_cfg["args"][0]

print(f"Server key   : {server_key}")
print(f"MCP endpoint : {MCP_ENDPOINT}")

Server key   : Swiftlet
MCP endpoint : http://localhost:3001/mcp/


## 3. Connect to MCP Server & Discover Tools

In [3]:
import httpx
from mcp_client import McpClient

mcp: McpClient | None = None
discovered_tools: list[dict] = []

print(f"Connecting to MCP server at: {MCP_ENDPOINT}")
print("-" * 50)

try:
    mcp = McpClient(endpoint=MCP_ENDPOINT, timeout_seconds=10.0)
    mcp.initialize()
    discovered_tools = mcp.list_tools()
    print(f"✓ Connected successfully")
    print(f"  Discovered {len(discovered_tools)} tool(s):")
    for t in discovered_tools:
        name = t.get("name", "unnamed")
        desc = t.get("description", "")[:80]
        print(f"    • {name}: {desc}")
except httpx.ConnectError:
    print(f"✗ Cannot reach MCP server at {MCP_ENDPOINT}")
    print("  → Make sure Rhino is running and Swiftlet is active on port 3001")
    mcp = None
except Exception as exc:
    print(f"✗ MCP error: {exc}")
    mcp = None

Connecting to MCP server at: http://localhost:3001/mcp/
--------------------------------------------------
✓ Connected successfully
  Discovered 3 tool(s):
    • site_boundary_generation: Generates procedural site boundaries using area, proportion, distortion, subdivi
    • Usable_site_area: area of Usable site area
    • parametric_shape_generator: Generates building shape boundaries using area, proportion, distortion, subdivis


## 4. Call `parametric_shape_generator` with Default Parameters

Directly invoke the tool — no LLM in the loop.

In [16]:
TOOL_NAME = "parametric_shape_generator"

# ── Adjust any parameter you want to test ────────────────────────────────────
TOOL_ARGS = {
    "shape_type": "L",
    "units": "meters",
    "parameters": {
        "arm_a_length":   32.0,
        "arm_b_length":   24.0,
        "building_width": 10.0,
        "courtyard_size": 18.0,
        "rotation_angle":  0.0,
        "base_point":     [0.0, 0.0, 0.0],
    },
}

print(f"Calling tool : {TOOL_NAME}")
print(f"Arguments    :")
print(json.dumps(TOOL_ARGS, indent=2))
print("-" * 50)

if mcp is None:
    print("✗ No MCP connection — skipping live call")
else:
    registered = {t.get("name") for t in discovered_tools}
    if TOOL_NAME not in registered:
        print(f"✗ Tool '{TOOL_NAME}' not found on server")
        print(f"  Available: {sorted(registered)}")
    else:
        try:
            raw = mcp.call_tool(TOOL_NAME, TOOL_ARGS)
            print(f"✓ Raw response:\n{raw}")

            try:
                result = json.loads(raw)
                meta = result.get("shape_metadata") or result.get("output", {}).get("shape_metadata", {})
                geo  = result.get("live_geometry")  or result.get("output", {}).get("live_geometry", {})
                if meta:
                    print(f"\nParsed metadata:")
                    print(f"  shape_type          : {meta.get('shape_type')}")
                    print(f"  gross_floorplate_area: {meta.get('gross_floorplate_area')} m²")
                    print(f"  courtyard_area      : {meta.get('courtyard_area')} m²")
                    print(f"  perimeter_length    : {meta.get('perimeter_length')} m")
                    print(f"  centroid            : {meta.get('centroid')}")
                    print(f"  bounding_box        : {meta.get('bounding_box')}")
                if geo:
                    coords = geo.get("coordinates", [])
                    print(f"  vertices            : {len(coords)}")
            except json.JSONDecodeError:
                pass  # raw already printed above

        except Exception as exc:
            print(f"✗ Tool call failed: {exc}")

Calling tool : parametric_shape_generator
Arguments    :
{
  "shape_type": "L",
  "units": "meters",
  "parameters": {
    "arm_a_length": 32.0,
    "arm_b_length": 24.0,
    "building_width": 10.0,
    "courtyard_size": 18.0,
    "rotation_angle": 0.0,
    "base_point": [
      0.0,
      0.0,
      0.0
    ]
  }
}
--------------------------------------------------
✓ Raw response:
L


In [17]:
TOOL_NAME = "parametric_shape_generator"

# ── Adjust any parameter you want to test ────────────────────────────────────
TOOL_ARGS = {
    "shape_type": "I",
    "units": "meters",
    "parameters": {
        "arm_a_length":   32.0,
        "arm_b_length":   24.0,
        "building_width": 10.0,
        "courtyard_size": 18.0,
        "rotation_angle":  0.0,
        "base_point":     [0.0, 0.0, 0.0],
    },
}

print(f"Calling tool : {TOOL_NAME}")
print(f"Arguments    :")
print(json.dumps(TOOL_ARGS, indent=2))
print("-" * 50)

if mcp is None:
    print("✗ No MCP connection — skipping live call")
else:
    registered = {t.get("name") for t in discovered_tools}
    if TOOL_NAME not in registered:
        print(f"✗ Tool '{TOOL_NAME}' not found on server")
        print(f"  Available: {sorted(registered)}")
    else:
        try:
            raw = mcp.call_tool(TOOL_NAME, TOOL_ARGS)
            print(f"✓ Raw response:\n{raw}")

            try:
                result = json.loads(raw)
                meta = result.get("shape_metadata") or result.get("output", {}).get("shape_metadata", {})
                geo  = result.get("live_geometry")  or result.get("output", {}).get("live_geometry", {})
                if meta:
                    print(f"\nParsed metadata:")
                    print(f"  shape_type          : {meta.get('shape_type')}")
                    print(f"  gross_floorplate_area: {meta.get('gross_floorplate_area')} m²")
                    print(f"  courtyard_area      : {meta.get('courtyard_area')} m²")
                    print(f"  perimeter_length    : {meta.get('perimeter_length')} m")
                    print(f"  centroid            : {meta.get('centroid')}")
                    print(f"  bounding_box        : {meta.get('bounding_box')}")
                if geo:
                    coords = geo.get("coordinates", [])
                    print(f"  vertices            : {len(coords)}")
            except json.JSONDecodeError:
                pass  # raw already printed above

        except Exception as exc:
            print(f"✗ Tool call failed: {exc}")

Calling tool : parametric_shape_generator
Arguments    :
{
  "shape_type": "I",
  "units": "meters",
  "parameters": {
    "arm_a_length": 32.0,
    "arm_b_length": 24.0,
    "building_width": 10.0,
    "courtyard_size": 18.0,
    "rotation_angle": 0.0,
    "base_point": [
      0.0,
      0.0,
      0.0
    ]
  }
}
--------------------------------------------------
✓ Raw response:
I


## 5. Call `parametric_shape_generator` with Custom Parameters

Test with different dimensions and a rotation angle — edit any value below.

In [ ]:
# ── Edit any values to explore different geometries ──────────────────────────
CUSTOM_ARGS = {
    "shape_type": "T_shape",
    "units": "meters",
    "parameters": {
        "arm_a_length":   40.0,
        "arm_b_length":   30.0,
        "building_width": 12.0,
        "courtyard_size": 20.0,
        "rotation_angle": 45.0,
        "base_point":     [10.0, 5.0, 0.0],
    },
}

print(f"Calling tool : {TOOL_NAME}")
print(f"Arguments    :")
print(json.dumps(CUSTOM_ARGS, indent=2))
print("-" * 50)

if mcp is None:
    print("✗ No MCP connection — skipping live call")
else:
    registered = {t.get("name") for t in discovered_tools}
    if TOOL_NAME not in registered:
        print(f"✗ Tool '{TOOL_NAME}' not found on server")
    else:
        try:
            raw = mcp.call_tool(TOOL_NAME, CUSTOM_ARGS)
            print(f"✓ Raw response:\n{raw}")

            try:
                result = json.loads(raw)
                meta = result.get("shape_metadata") or result.get("output", {}).get("shape_metadata", {})
                geo  = result.get("live_geometry")  or result.get("output", {}).get("live_geometry", {})
                if meta:
                    print(f"\nParsed metadata:")
                    print(f"  shape_type          : {meta.get('shape_type')}")
                    print(f"  gross_floorplate_area: {meta.get('gross_floorplate_area')} m²")
                    print(f"  courtyard_area      : {meta.get('courtyard_area')} m²")
                    print(f"  perimeter_length    : {meta.get('perimeter_length')} m")
                    print(f"  centroid            : {meta.get('centroid')}")
                    print(f"  rotation_angle      : {meta.get('rotation_angle')}°")
                if geo:
                    coords = geo.get("coordinates", [])
                    print(f"  vertices            : {len(coords)}")
            except json.JSONDecodeError:
                pass

        except Exception as exc:
            print(f"✗ Tool call failed: {exc}")

## 6. Batch Test — All Shape Types

Calls `parametric_shape_generator` for every shape type with the same base parameters and summarises the results.

In [ ]:
ALL_SHAPE_TYPES = ["L_shape", "Y_shape", "T_shape", "H_shape", "X_shape", "Z_shape", "O_shape"]

BASE_PARAMETERS = {
    "units": "meters",
    "parameters": {
        "arm_a_length":   32.0,
        "arm_b_length":   24.0,
        "building_width": 10.0,
        "courtyard_size": 18.0,
        "rotation_angle":  0.0,
        "base_point":     [0.0, 0.0, 0.0],
    },
}

print("=" * 70)
print("BATCH TEST — All Shape Types via MCP")
print("=" * 70)

if mcp is None:
    print("✗ No MCP connection — cannot run batch test")
else:
    registered = {t.get("name") for t in discovered_tools}
    if TOOL_NAME not in registered:
        print(f"✗ Tool '{TOOL_NAME}' not found on server")
    else:
        results_summary = []
        for shape in ALL_SHAPE_TYPES:
            args = {"shape_type": shape, **BASE_PARAMETERS}
            try:
                raw = mcp.call_tool(TOOL_NAME, args)
                result = json.loads(raw) if raw.strip().startswith("{") else {}
                meta = result.get("shape_metadata") or result.get("output", {}).get("shape_metadata", {})
                if meta:
                    status = "✓"
                    area  = meta.get("gross_floorplate_area", "—")
                    perim = meta.get("perimeter_length", "—")
                    geo   = result.get("live_geometry") or result.get("output", {}).get("live_geometry", {})
                    verts = len(geo.get("coordinates", [])) if geo else "—"
                else:
                    status = "✗"
                    area = perim = verts = "—"
            except Exception as exc:
                status = "✗"
                area = perim = verts = str(exc)[:30]

            results_summary.append((shape, status, area, perim, verts))
            print(f"  [{shape:<10}] {status}  area={str(area):>8}m²  perim={str(perim):>8}m  verts={verts}")

        passed = sum(1 for r in results_summary if r[1] == "✓")
        print(f"\n{passed}/{len(ALL_SHAPE_TYPES)} shapes returned successfully")

## 7. Cleanup

Close the MCP HTTP client when done.

In [ ]:
if mcp is not None:
    mcp.close()
    print("MCP client closed.")
else:
    print("No active MCP client to close.")